In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Data/UCI_Credit_Card.csv")
df = pd.get_dummies(df, columns=["SEX"], dtype=int)
df = pd.get_dummies(df, columns=["EDUCATION"], dtype=int)
df = pd.get_dummies(df, columns=["MARRIAGE"], dtype=int)
pay_columns = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
df = pd.get_dummies(df, columns=pay_columns, dtype=int)

x = df.drop(columns=["ID", "default.payment.next.month"])
y = df["default.payment.next.month"]

In [2]:
import torch
from torch.utils.data import TensorDataset, random_split, DataLoader

# pandas → Tensor
# pytorch用tensor做運算，所以需要將pandas轉為tensor
x_tensor = torch.tensor(x.to_numpy(), dtype=torch.float32)
y_tensor = torch.tensor(
    y.to_numpy(), dtype=torch.float32
).reshape(-1, 1)  #把y也改成2 dim，  

# 將x、y配對
# dataset[0]會取得(x_tensor[0], y_tensor[0])
dataset = TensorDataset(x_tensor, y_tensor)

# 切分80%訓練集、20%驗證集
n_val = int(len(dataset) * 0.2)  #資料筆數不能是小數，所以轉成整數，小數部分會被捨去
n_train = len(dataset) - n_val

train_dataset, val_dataset = random_split(
    dataset,
    [n_train, n_val],  #第一組要有幾筆、第二組要有幾筆
    generator=torch.Generator().manual_seed(18)
)

# 分批載入
train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=64, shuffle=False
)

In [3]:
import random
from torch import nn

# 固定隨機種子，讓結果較容易重現
random.seed(18)
np.random.seed(18)
torch.manual_seed(18)
torch.set_num_threads(2)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("目前使用的裝置：", device)

# 1. 標準化：只能使用訓練集計算平均值與標準差，因為驗證集是用來模擬「模型沒看過的資料」
# 直接把train dataset中的每個column都做scale了
train_x = x_tensor[train_dataset.indices]  # train_dataset.indices記錄了哪些原始資料column被分到訓練集
'''
# 假設訓練集選到原始資料的第 2、5、8 列
train_dataset.indices = [2, 5, 8]
# 概念上等同取出這三列
train_x = x_tensor[[2, 5, 8]]
'''

mean = train_x.mean(dim=0) # dim=0沿著column的方向
std = train_x.std(dim=0, unbiased=False)  # unbiased=False表示計算變異數時，分母使用資料筆數N，而不是N−1

# 避免某些欄位標準差為0，造成除以0，太小的設為1.0
std[std < 1e-6] = 1.0


# 2. 建立 MLP
class MLP(nn.Module):
    def __init__(self, input_dim, mean, std): #input_dim輸入特徵有多少個
        super().__init__()  #super().__init__()會先執行父類別nn.Module的初始化

        # 將標準化參數存進模型，但不當成需要訓練的權重
        self.register_buffer("mean", mean)
        self.register_buffer("std", std)

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, inputs):  #定義資料通過模型的流程
        # 放在模型內，訓練、驗證與單筆預測都會自動標準化
        inputs = (inputs - self.mean) / self.std
        return self.net(inputs)


model = MLP(x_tensor.shape[1], mean, std).to(device) #x_tensor.shape[1]是取column數量

# 二元分類：直接接收 logits，因此模型最後不要加 Sigmoid
loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

目前使用的裝置： cuda


In [4]:
import copy

epochs = 50
train_losses = []
val_losses = []

best_val_loss = float("inf")
best_epoch = 0
best_state = None


# 3. 訓練完整 50 epochs
for epoch in range(epochs):
    model.train()  # 訓練模式：啟用 Dropout
    train_sum = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()       # 清除上一批的梯度
        logits = model(xb)           # 前向傳播
        loss = loss_function(logits, yb)
        loss.backward()             # 計算梯度
        optimizer.step()            # 更新權重

        train_sum += loss.item() * xb.size(0)

    # 驗證：不計算梯度，也不更新權重，並關閉 Dropout
    model.eval()  # 驗證模式：eval() 不會關閉梯度計算，所以後面還會搭配 torch.no_grad()
    val_sum = 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = loss_function(logits, yb)
            val_sum += loss.item() * xb.size(0)

    # 用樣本數加權，正確處理最後不足一個 batch 的情況
    train_loss = train_sum / len(train_dataset)
    val_loss = val_sum / len(val_dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # 保存驗證損失最低的權重，但仍繼續跑滿 50 epochs
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())  #存下來

    print(
        f"Epoch {epoch + 1:02d}/{epochs} | "
        f"Training Loss={train_loss:.4f} | "
        f"Validation Loss={val_loss:.4f}"
    )

Epoch 01/50 | Training Loss=0.4695 | Validation Loss=0.4423
Epoch 02/50 | Training Loss=0.4362 | Validation Loss=0.4422
Epoch 03/50 | Training Loss=0.4348 | Validation Loss=0.4370
Epoch 04/50 | Training Loss=0.4308 | Validation Loss=0.4392
Epoch 05/50 | Training Loss=0.4293 | Validation Loss=0.4374
Epoch 06/50 | Training Loss=0.4300 | Validation Loss=0.4359
Epoch 07/50 | Training Loss=0.4283 | Validation Loss=0.4358
Epoch 08/50 | Training Loss=0.4258 | Validation Loss=0.4355
Epoch 09/50 | Training Loss=0.4270 | Validation Loss=0.4376
Epoch 10/50 | Training Loss=0.4261 | Validation Loss=0.4367
Epoch 11/50 | Training Loss=0.4247 | Validation Loss=0.4345
Epoch 12/50 | Training Loss=0.4260 | Validation Loss=0.4341
Epoch 13/50 | Training Loss=0.4230 | Validation Loss=0.4340
Epoch 14/50 | Training Loss=0.4228 | Validation Loss=0.4362
Epoch 15/50 | Training Loss=0.4233 | Validation Loss=0.4365
Epoch 16/50 | Training Loss=0.4221 | Validation Loss=0.4359
Epoch 17/50 | Training Loss=0.4214 | Val

In [ ]:
import matplotlib.pyplot as plt

# 4. 繪製 Training Loss 與 Validation Loss
fig = plt.figure(figsize=(9, 5))

plt.plot(range(1, epochs + 1), train_losses, label="Training Loss")
plt.plot(range(1, epochs + 1), val_losses, label="Validation Loss")

plt.axvline(
    best_epoch,
    color="gray",
    linestyle="--",
    label=f"Best epoch: {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("MLP Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig("../Data/training_validation_loss.png", dpi=300, bbox_inches="tight")
plt.close(fig)


In [6]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 載入 Validation Loss 最低的模型
model.load_state_dict(best_state)
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)

        logits = model(xb)
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).long()

        all_labels.extend(yb.reshape(-1).cpu().tolist())
        all_predictions.extend(predictions.reshape(-1).cpu().tolist())

print(f"使用第 {best_epoch} 個 epoch 的模型")

accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(
    all_labels, all_predictions, pos_label=1, zero_division=0
)
recall = recall_score(
    all_labels, all_predictions, pos_label=1, zero_division=0
)
f1 = f1_score(
    all_labels, all_predictions, pos_label=1, zero_division=0
)

print(f"Accuracy ：{accuracy:.2%}")
print(f"Precision：{precision:.2%}")
print(f"Recall   ：{recall:.2%}")
print(f"F1-score ：{f1:.2%}")

使用第 13 個 epoch 的模型
Accuracy ：81.95%
Precision：69.71%
Recall   ：36.84%
F1-score ：48.21%


In [7]:
# 5. 實際預測 Validation Dataset的一筆sample
model.load_state_dict(best_state)
model.eval()

sample_x, sample_y = val_dataset[35]

with torch.no_grad():
    # 原本 shape = [特徵數]
    # 加上 batch 維度後 = [1, 特徵數]
    sample_input = sample_x.unsqueeze(0).to(device)

    logit = model(sample_input)
    probability = torch.sigmoid(logit).item()

predicted_label = int(probability >= 0.5)
actual_label = int(sample_y.item())

print(f"預測違約機率：{probability:.2%}")
print(f"預測類別：{predicted_label}")
print(f"真實類別：{actual_label}")
print("0 = 未違約，1 = 違約")

預測違約機率：2.11%
預測類別：0
真實類別：0
0 = 未違約，1 = 違約


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

model.load_state_dict(best_state)
model.eval()

all_labels = []
all_scores = []

# 取得 Validation Dataset 的真實標籤與預測機率
with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb.to(device))
        probabilities = torch.sigmoid(logits)

        all_labels.extend(yb.reshape(-1).cpu().tolist())
        all_scores.extend(probabilities.reshape(-1).cpu().tolist())

y_true = np.array(all_labels, dtype=int)
y_score = np.array(all_scores)

# ROC 使用連續機率，不要先用 0.5 閾值轉成預測類別
fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
auc_value = roc_auc_score(y_true, y_score)

print("前 10 筆預測違約機率：", y_score[:10])
print(f"MLP Validation AUC：{auc_value:.4f}")

# 繪製 ROC Curve，並在圖例標示 AUC
fig = plt.figure(figsize=(9, 5))
plt.plot(fpr, tpr, linewidth=2, label=f"MLP (AUC = {auc_value:.4f})")
plt.plot([0, 1], [0, 1], "k--", label="Random Classifier (AUC = 0.5)")

plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("MLP ROC Curve - Validation Dataset")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../Data/ROC_curve.png", dpi=300, bbox_inches="tight")
plt.close(fig)

前 10 筆預測違約機率： [0.18468374 0.11716998 0.05686492 0.68199337 0.30587679 0.18058813
 0.22065859 0.57613546 0.11229154 0.11603887]
MLP Validation AUC：0.7806
